## Read .data.info binary file

This notebook reads the binary `.data.info` files generated by the `MPISAVEX` subroutine in the Fortran code. These files contain metadata about the simulation.

### Configuration

A common issue when reading Fortran binary files is a mismatch in integer size or byte order (endianness). Use the cell below to configure the script to match your Fortran compiler's settings.

In [ ]:
# ==> CONFIGURATION <==
# Endianness: 'big' or 'little'
ENDIANNESS = 'little'

# Integer size in bytes: 4 or 8
INTEGER_SIZE = 4
# Automatically set format characters based on above settings
ENDIAN_PREFIX = '>' if ENDIANNESS == 'big' else '<'
INTEGER_FORMAT = 'i' if INTEGER_SIZE == 4 else 'q'

### Helper Functions

In [ ]:
import struct
import os

In [ ]:
def read_fortran_record(f):
    """Reads a Fortran unformatted record with configurable endianness and integer size."""
    try:
        record_len_bytes = f.read(INTEGER_SIZE)
        if not record_len_bytes:
            return None, 0

        if len(record_len_bytes) < INTEGER_SIZE:
            print(f'DEBUG: Could not read a full {INTEGER_SIZE}-byte record length. Got only {len(record_len_bytes)} bytes.')
            return None, 0

        record_len_format = ENDIAN_PREFIX + INTEGER_FORMAT
        record_len = struct.unpack(record_len_format, record_len_bytes)[0]

        data = f.read(record_len)
        if len(data) < record_len:
            print(f'DEBUG: Tried to read {record_len} bytes of data, but only got {len(data)}.')
            return None, 0

        # Read the end-of-record marker
        f.read(INTEGER_SIZE)

        return data, record_len

    except (struct.error, IOError) as e:
        print(f'ERROR reading Fortran record: {e}')
        return None, 0

In [ ]:
def read_data_info(file_path):
    """Reads a .data.info file and returns a dictionary of the contents."""
    info = {}
    print(f'-- Config: Endianness={ENDIANNESS}, Integer Size={INTEGER_SIZE} bytes --')
    try:
        with open(file_path, 'rb') as f:
            # Record 1: NDIMR, NDIMTH, NDIMX
            data, length = read_fortran_record(f)
            if data is None: return None
            info['NDIMR'], info['NDIMTH'], info['NDIMX'] = struct.unpack(ENDIAN_PREFIX + 'iii', data)
            
            # Record 2: NRCHOPDIM, NTCHOPDIM, NXCHOPDIM
            data, length = read_fortran_record(f)
            if data is None: return None
            info['NRCHOPDIM'], info['NTCHOPDIM'], info['NXCHOPDIM'] = struct.unpack(ENDIAN_PREFIX + 'iii', data)
            
            # Record 3: local_scalar%SPACE, local_scalar%LN
            data, length = read_fortran_record(f)
            if data is None: return None
            info['SPACE'], info['LN'] = struct.unpack(ENDIAN_PREFIX + 'id', data)
            
            # Record 4: ZLEN, ELL
            data, length = read_fortran_record(f)
            if data is None: return None
            info['ZLEN'], info['ELL'] = struct.unpack(ENDIAN_PREFIX + 'dd', data)
            
            # Record 5: MINC, MKLINK
            data, length = read_fortran_record(f)
            if data is None: return None
            info['MINC'], info['MKLINK'] = struct.unpack(ENDIAN_PREFIX + 'ii', data)

    except (IOError, struct.error) as e:
        print(f'An error occurred during unpacking: {e}')
        return None
        
    print('-- Successfully read all records. --')
    return info

### Execution

In [ ]:
file_path = '../../output/b0.dat.info'

if os.path.exists(file_path):
    info_data = read_data_info(file_path)
    if info_data:
        print('--- Parsed Data ---')
        for key, value in info_data.items():
            print(f'{key}: {value}')
else:
    print(f'File not found: {file_path}')
    print('Please update the file_path variable to point to a valid .data.info file.')